# Stage1 尾盘ROD：来源资格与固定策略检验

第一阶段已完成来源预检：只核来源、日历、事件覆盖、端点与Qlib表示误差。下方第二阶段仅在完整策略独评和owner GO后执行；两阶段证据分开记录。


In [1]:
import os, sys, json
from pathlib import Path
from IPython.display import display, Markdown
root = Path.cwd()
while not (root / "AGENTS.md").exists():
    if root.parent == root: raise RuntimeError("workspace not found")
    root = root.parent
sys.path.insert(0, str(root))
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"
import qlib
from qlib.data import D
from qlib.workflow import R
from src.workflow.close_rod_source_preflight import run
card = root / "research/stage1_close_rod_20261018"
# run uses qlib.init, D.calendar/D.features, DatasetH.prepare and R.start.
result = run(root=root, card=card)
result

{'experiment': 'stage1_close_rod_20261018_source_preflight',
 'plan_sha256': '1f11bbdaac7d01f739820ca325390217d718d2dc252f21094e961cb0b8f36c46',
 'stage': 'source_only_preflight_completed',
 'decision': 'source_dependencies_observed_for_separate_strategy_review',
 'endpoint_gate_pass': True,
 'account_sessions': 944,
 'regular_sessions': 936,
 'half_days': 8,
 'first_previous_session': '2023-01-03',
 'first_previous_close': '2023-01-03 16:00:00',
 'prior_half_day_closes_used': 8,
 'endpoint_rows': 15060,
 'invalid_endpoint_rows': 0,
 'native_float64_rows': 11302,
 'original_anchor_checks': [{'instrument': 'SOXL',
   'original_points': 3778,
   'exact_float32_anchor_equality': True},
  {'instrument': 'SOXX',
   'original_points': 3778,
   'exact_float32_anchor_equality': True}],
 'event_audit': {'status': 'conditional_snapshot_event_mask_ready',
  'qualification': 'frozen_current_history_non_event_classification_not_historical_PIT',
  'event_tail_supported': True,
  'qualified_through':

## 中文来源结论

仅来源字段，0signal/0label/0fit/0policy/0account。正式策略设计及执行仍需独评和owner GO；不把来源可用性当稳定超额。

来源预检完成：944账户日，其中936正常日、8半日。必要端点15060行，缺失或无效0行；两资产各3778个原账户价格锚点精确核对。首个上一交易日为2023-01-03，实际收盘2023-01-03 16:00:00；前一半日13:00收盘按原日历使用。

事件源当前快照覆盖至2026-10-09，尾段支持=True。事件及未知覆盖日双方均不允许新增尾盘lot，完整原账户日保留；这不构成历史PIT的无事件证明。

仅校验原始价格和Qlib数值表示误差，未比较两价格生成方向，未计算ROD/尾盘收益/标签，0fit/0policy/0account/0新市场数据HTTP。float64价格是未来获批后的严格符号权威；Qlibfloat32仅派生缓存。

决定：source_dependencies_observed_for_separate_strategy_review。来源与NAS身份见source_preflight/summary.json；正式策略设计独评和另行执行GO仍必需。稳定超额目标未完成。

独立来源结果评价已允许进入完整策略冻结；不批准ROD信号计算或策略账户。NAS冷恢复28个产物哈希、原生float64 Dataset、端点与事件mask均一致。旧锚点保持原F32投影，新增成交/估值也用F32读数升64算账；未来ROD符号另用已验证native float64严格价格比较。来源PIT和独立前向资格仍为false。详见source_result_review.json与source_preflight/readback_verification.json。


## 完整策略冻结与无方向预检

保留上方已完成来源结论。这里只构建原3778+新增2808个共同价格点、核引用身份；不计算真实ROD方向或策略机会。新许可与lot规则见[plan.json](plan.json)。

In [2]:
from src.workflow.close_rod import prepare, run as run_strategy
prepared_strategy = prepare(root=root, card=card)
prepared_strategy["receipt"]

{'plan_sha256': 'a548cbac5bf81787d9b14cc4491c711c903476489f17488b98e5e638bb9387b1',
 'account_sessions': 944,
 'regular_sessions': 936,
 'half_days': 8,
 'source_regular_non_event_sessions': 908,
 'old_anchor_points': 3778,
 'expanded_grid_points': 6586,
 'old_anchor_values_exact': True,
 'new_quote_plane': 'native D.features float32 promoted tofloat64',
 'signal_plane': 'verified native float64; not compared',
 'reference_objects_byte_verified': 12,
 'reference_objects_loaded_or_valued': 0,
 'new_ROD_values_or_signs': 0,
 'new_policy_paths': 0,
 'new_native_accounts': 0,
 'new_model_fits': 0,
 'source_audits': {'base': {'recorder_id': '6e227278b7a64eafbcf34b965e16d2b9',
   'experiment': 'stage1_domain_heads_20261018',
   'experiment_id': '283510277269468590',
   'recorder_uri': '/Users/rziiiii/dev/workspace/research/stage1_domain_heads_20261018/qlib_recorder',
   'recorder_local_dir': '/Users/rziiiii/dev/workspace/research/stage1_domain_heads_20261018/qlib_recorder/283510277269468590/

## 正式策略执行门

仅CLOSE_ROD_EXECUTE=1且完整独评/owner approval精确匹配计划和代码时计算float64方向并执行6新账户与12被动参考估值。source-only批准不适用此门。

In [3]:
if os.environ.get("CLOSE_ROD_EXECUTE") == "1":
    strategy_result = run_strategy(root=root, card=card, prepared=prepared_strategy)
else:
    strategy_result = {"status": "not_executed_pending_strategy_approval", "signals": 0, "new_native_accounts": 0, "passive_valuations": 0}
strategy_result

completed native B_close_permission_0
completed native B_close_permission_2
completed native B_close_permission_5
completed native C_close_ROD_positive_0
completed native C_close_ROD_positive_2
completed native C_close_ROD_positive_5


Qlib emitted its repeated RuntimeWarning: Mean of empty slice 5676 times; duplicates compacted. Native reconciliation remains in Recorder.


{'experiment': 'stage1_close_rod_20261018',
 'plan_sha256': 'a548cbac5bf81787d9b14cc4491c711c903476489f17488b98e5e638bb9387b1',
 'decision': 'not_promoted_fixed_close_ROD_v1',
 'gates': {'net_relative_wealth_all_costs': False,
  'sampled_drawdown_30pct': True,
  'cvar_no_worse_soxx': True,
  'minimum_complete_years': True,
  'complete_annual_70pct': False,
  'rolling126_70pct': False,
  'paired_block_lower_positive': False,
  'fresh_fullminute_risk_qualified': False,
  'qualified_independent_forward': False},
 'coarse_economic_gates_pass': False,
 'no_executable_lots': False,
 'objective_achieved': False,
 'new_fits': 0,
 'new_policy_paths': 6,
 'new_native_accounts': 6,
 'reused_native_reference_accounts': 12,
 'passive_reference_revaluations': 12,
 'total_comparison_accounts': 18,
 'reference_added_marks_native_validated': False,
 'sessions_each': 944,
 'points_each': 6586,
 'recorder_id': 'f1f70f58b59946feabe131ff8b3d9645',
 'experiment_id': '744225113974357313',
 'recorder_uri': '/

In [4]:
if os.environ.get("CLOSE_ROD_EXECUTE") == "1":
    display(Markdown((card / "conclusion.md").read_text()))

# 尾盘ROD新增受限恢复许可：固定结果

原f696+b472晨间链保留；B/C共享新增尾盘许可，C仅多严格float64价格方向门。0拟合、6新原生账户；12旧native参考只被动增加估值点，不能称作12次新native验证。

|方案|滑点bp|净收益%|收盘DD%|全采样点DD%|
|---|---:|---:|---:|---:|
|A_active_research_replay|0|229.686|-29.487|-29.717|
|A_active_research_replay|2|280.650|-27.060|-27.266|
|A_active_research_replay|5|41.521|-30.517|-31.539|
|B_close_permission|0|267.165|-29.175|-29.411|
|B_close_permission|2|137.206|-27.666|-27.870|
|B_close_permission|5|177.288|-27.900|-28.103|
|SOXX|0|380.588|-40.863|-42.926|
|SOXX|2|380.568|-40.867|-42.929|
|SOXX|5|380.538|-40.872|-42.935|
|strong9097|0|441.681|-26.812|-26.812|
|strong9097|2|434.269|-26.881|-26.881|
|strong9097|5|423.730|-26.956|-26.956|
|C_close_ROD_positive|0|183.479|-29.386|-29.617|
|C_close_ROD_positive|2|138.856|-27.269|-27.474|
|C_close_ROD_positive|5|29.057|-27.447|-27.652|
|additional27baa|0|485.498|-27.171|-27.171|
|additional27baa|2|477.826|-27.223|-27.223|
|additional27baa|5|466.073|-27.317|-27.317|

决定：not_promoted_fixed_close_ROD_v1。失败或未获得的门：net_relative_wealth_all_costs, complete_annual_70pct, rolling126_70pct, paired_block_lower_positive, fresh_fullminute_risk_qualified, qualified_independent_forward。

全部年/季度（含partial）、126日滚动及12项置信下界均保留，不择窗；费用情景各自NAV/风险/费用状态独立。新增整数lot按15:59精确退出，不重置财富。来源为条件历史snapshot，假设延迟并非实盘首收或auction成交证明，重复历史不是前向独立证据。

若没有可执行整数lot，应解释为机会集无信息，不能声称检验出ROD无效；固定版本失败则关闭，不改符号、窗口、caps、时钟或退出参数救援。执行者不自签promotion，等待独立结果评估。

Recorder：`f1f70f58b59946feabe131ff8b3d9645`。来源结论独立保存在source_preflight/summary.json；稳定超额目标未完成。

固定结果与区间差异：C在0/2/5bp的累计净收益为183.479%/138.856%/29.057%；2bp B为137.206%、A为280.650%、SOXX为380.568%。C只在2bp略胜B，不能通过全部成本对照。12项预定置信下界全部失败；30%采样回撤和CVaR门通过，但全窗财富、年度和rolling门失败，版本关闭。

完整2024/2025年度，C的2bp净收益分别−8.050%/18.324%，相对SOXX财富0.783028/0.823463。计入两个partial年度后，2bp C胜B为3/4、胜A及SOXX均0/4；全部16季度分别6/16、3/16、6/16。完整14季度及partial2季度单列，全部成本的180项配对完整保留于failure_period_comparison.csv。

一次固定归因并非机会空集：B在0/2/5bp实际尾盘交易日142/207/168，C为92/113/141。所有执行整数lot均精确平掉，累计股数恒等式误差0，现金恒等式最大误差约5.27e-11美元。2bp C已执行lot毛价差−158.71美元，entry+exit费用1939.72美元，现金净变动−2098.43美元；其对A终值差却为−141794.37美元。这是不同层面的已记录数值，不能把后者简单归于这笔费用或单一controller。

2bp C与B的risk_state/risk_clock在944日完全相同，但open执行决定18日不同，首次为2023-08-16。C与A的risk_state在241日不同，首日2024-02-21；clock不同274日，open执行不同58日。C三成本0对2、0对5、2对5之间，risk_state分别有170、304、194日不同。跨成本差异包括原Stage1自身费用/状态敏感性和新增尾盘反馈，不是简单费用相减。

2bp期末端点平均SOXL/SOXX/cash权重：C为12.388%/35.777%/51.835%，B为12.264%/35.646%/52.089%，A为15.082%/36.597%/48.321%。这是next-open端点持仓的描述，不是日内时间平均暴露或因果cash-drag分解。机会拒绝原因按原固定顺序记录第一个失败条件，且来自各自状态；详见failure_late_reasons.csv，不将其解释成共同反事实机会集。

一次交付冷读已由研究owner完成，见[readback_verification.json](readback_verification.json)：82个artifact、sign Dataset及6native/12passive角色核验。执行者没有重复NAS冷读或替自己签署晋升；固定负结果仍交独立结果评估。一次归因来源和公式见[failure_attribution.json](failure_attribution.json)。整个稳定超额Goal仍未达成。


## 一次固定失败归因

归因scope先保存为failure_attribution_plan.json；随后独立短notebook执行session10854、exit0。只读已存本轮Recorder与冻结A的native authority；未执行新fit、policy/native回放、CI重抽、NAS冷读或source请求。


In [1]:
import os, json, hashlib
from pathlib import Path
import numpy as np
import pandas as pd
import qlib
from src.core.recorder_sources import load_verified_recorder_objects
root=Path('/Users/rziiiii/dev/workspace');card=root/'research/stage1_close_rod_20261018'
plan=json.loads((card/'plan.json').read_text());summary=json.loads((card/'summary.json').read_text())
manifest=json.loads((card/'verification_manifest.json').read_text())
qlib.init(provider_uri=summary['provider_uri'],region='us',kernels=1)
keys=['inputs/late_inputs','analysis/metrics','analysis/annual_metrics','analysis/quarterly_metrics','analysis/uncertainty']
keys += [f'native/{family}_{cost}' for family in ['B_close_permission','C_close_ROD_positive'] for cost in [0,2,5]]
hashes={key:next(x['sha256'] for x in manifest['entries'] if x['path'].endswith('/artifacts/'+key)) for key in keys}
objects,audit=load_verified_recorder_objects(card,expected_recorder_id=summary['recorder_id'],artifacts=keys,
    expected_manifest_sha256=hashlib.sha256((card/'verification_manifest.json').read_bytes()).hexdigest(),expected_artifact_sha256=hashes)
a_binding=plan['reference_sources']['active'];a_card=root/a_binding['card']
assert json.loads((a_card/'nas_binding.json').read_text())['evidence']['sha256']==a_binding['archive']
a_objects,a_audit=load_verified_recorder_objects(a_card,expected_recorder_id=a_binding['recorder_id'],
    artifacts=a_binding['artifacts'],expected_manifest_sha256=a_binding['inventory_sha256'],expected_artifact_sha256=a_binding['artifact_sha256'])
books={(family,cost):objects[f'native/{family}_{cost}'] for family in ['B_close_permission','C_close_ROD_positive'] for cost in [0,2,5]}
books.update({('A_active_research_replay',cost):a_objects[f'native/A_active_research_replay_{cost}'] for cost in [0,2,5]})
# No future-return lookup for rejected opportunities: only recorded executed orders.
cash_rows=[];reason_rows=[]
for (family,cost),book in books.items():
    if family=='A_active_research_replay':continue
    events=book['late_events'];orders=book['orders']
    entry=orders.loc[orders.slot.eq('late_entry')];exit_orders=orders.loc[orders.slot.eq('late_exit')]
    assert not entry.sell.any() and exit_orders.sell.all()
    assert pd.DatetimeIndex(entry.stamp).strftime('%H:%M').isin(['15:32']).all()
    assert pd.DatetimeIndex(exit_orders.stamp).strftime('%H:%M').isin(['15:59']).all()
    assert events.lot_closed.all() and len(events)==944
    identity_errors=[];inventory_errors=[]
    for event in events.itertuples():
        bought=entry.loc[entry.trade_date.eq(event.trade_date)];sold=exit_orders.loc[exit_orders.trade_date.eq(event.trade_date)]
        flow=float(sold.notional.sum()-bought.notional.sum()-sold.total.sum()-bought.total.sum())
        identity_errors.append(abs(flow-float(event.roundtrip_pnl_usd)))
        for symbol,label in [('SOXL','soxl'),('SOXX','soxx')]:
            b=float(bought.loc[bought.symbol.eq(symbol),'raw_shares'].sum());s=float(sold.loc[sold.symbol.eq(symbol),'raw_shares'].sum())
            expected=float(getattr(event,'entry_shares_'+label));inventory_errors.extend([abs(b-s),abs(b-expected)])
        if pd.notna(getattr(event,'cash_before',np.nan)):
            identity_errors.append(abs(flow-(event.cash_after-event.cash_before)))
            inventory_errors.extend([abs(event.after_exit_soxl-event.baseline_soxl),abs(event.after_exit_soxx-event.baseline_soxx)])
    assert max(identity_errors)<1e-7 and max(inventory_errors)==0
    labels=[]
    for event in events.itertuples():
        label=event.reason
        if event.late_action:label='executed_lot'
        elif label=='eligible':
            known=float(getattr(event,'known_funded_soxl',0))+float(getattr(event,'known_funded_soxx',0))
            label='known_funding_or_envelope_zero' if known==0 else 'arrival_funding_or_envelope_zero'
        labels.append(label)
    counts=pd.Series(labels).value_counts()
    assert counts.sum()==944
    for reason,count in counts.items():reason_rows.append({'family':family,'slippage_bps':cost,'reason':reason,'days':int(count),
        'meaning':'first recorded failure after fixed predicate order; own-state opportunities, not a common counterfactual set'})
    row={'family':family,'slippage_bps':cost,'sessions':944,'recorded_source_ready_days':int(events.source_ready.fillna(False).astype(bool).sum()),
        'recorded_permission_pass_days_after_arm_gate':int(events.reason.eq('eligible').sum()),'late_trade_days':int(events.late_action.sum()),
        'entry_orders':len(entry),'exit_orders':len(exit_orders),'entry_notional_usd':float(entry.notional.sum()),
        'exit_notional_usd':float(exit_orders.notional.sum()),'entry_fee_usd':float(entry.total.sum()),'exit_fee_usd':float(exit_orders.total.sum()),
        'slippage_usd':float(entry.slippage.sum()+exit_orders.slippage.sum()),
        'gross_executed_lot_price_pnl_usd':float(exit_orders.notional.sum()-entry.notional.sum()),
        'net_recorded_lot_cash_pnl_usd':float(events.roundtrip_pnl_usd.sum()),
        'maximum_cash_identity_error_usd':max(identity_errors),'maximum_inventory_identity_error_shares':max(inventory_errors),
        'all_lots_closed':bool(events.lot_closed.all())}
    for symbol in ['SOXL','SOXX']:row['executed_roundtrip_shares_'+symbol]=int(entry.loc[entry.symbol.eq(symbol),'raw_shares'].sum())
    cash_rows.append(row)
cash=pd.DataFrame(cash_rows);cash.to_csv(card/'failure_late_cashbook.csv',index=False)
pd.DataFrame(reason_rows).to_csv(card/'failure_late_reasons.csv',index=False)

periods=pd.concat([objects['analysis/annual_metrics'],objects['analysis/quarterly_metrics']],ignore_index=True)
period_rows=[]
for comparator in ['B_close_permission','A_active_research_replay','SOXX']:
    fields=['kind','period','slippage_bps','complete_period','sessions','return_pct','terminal_wealth']
    left=periods.loc[periods.family.eq('C_close_ROD_positive'),fields]
    right=periods.loc[periods.family.eq(comparator),fields]
    joined=left.merge(right,on=['kind','period','slippage_bps'],suffixes=('_candidate','_reference'),validate='one_to_one')
    assert joined.complete_period_candidate.equals(joined.complete_period_reference)
    assert joined.sessions_candidate.equals(joined.sessions_reference)
    for row in joined.itertuples():
        relative=row.terminal_wealth_candidate/row.terminal_wealth_reference
        period_rows.append({'kind':row.kind,'period':row.period,'slippage_bps':row.slippage_bps,'complete_period':bool(row.complete_period_candidate),
            'sessions':row.sessions_candidate,'candidate':'C_close_ROD_positive','comparator':comparator,
            'candidate_return_pct':row.return_pct_candidate,'comparator_return_pct':row.return_pct_reference,
            'relative_wealth':relative,'candidate_wins':bool(relative>1+1e-8)})
period_table=pd.DataFrame(period_rows);assert len(period_table)==180
period_table.to_csv(card/'failure_period_comparison.csv',index=False)
period_summary=period_table.groupby(['kind','slippage_bps','comparator','complete_period'],as_index=False).agg(periods=('candidate_wins','size'),wins=('candidate_wins','sum'))
period_summary['positive_fraction']=period_summary.wins/period_summary.periods
period_summary.to_csv(card/'failure_period_summary.csv',index=False)

fields=['risk_state','risk_clock','execute_open_target','recovery_action']
def compare(left_key,right_key,role):
    left=books[left_key]['authority'];right=books[right_key]['authority']
    pd.testing.assert_frame_equal(left[['signal_date','trade_date','exit_date']],right[['signal_date','trade_date','exit_date']])
    assert left[fields].notna().all().all() and right[fields].notna().all().all()
    combined=np.zeros(len(left),dtype=bool)
    result={'role':role,'left_family':left_key[0],'left_cost':left_key[1],'right_family':right_key[0],'right_cost':right_key[1],'sessions':len(left)}
    for field in fields:
        different=left[field].ne(right[field]);combined|=different.to_numpy()
        result[field+'_different_days']=int(different.sum())
        result[field+'_first_difference']=str(pd.Timestamp(left.loc[different,'trade_date'].iloc[0]).date()) if different.any() else None
    result['any_recorded_policy_state_different_days']=int(combined.sum())
    result['first_recorded_policy_state_difference']=str(pd.Timestamp(left.loc[combined,'trade_date'].iloc[0]).date()) if combined.any() else None
    return result
pairs=[]
for cost in [0,2,5]:
    for left,right in [('B_close_permission','C_close_ROD_positive'),('B_close_permission','A_active_research_replay'),('C_close_ROD_positive','A_active_research_replay')]:
        pairs.append(compare((left,cost),(right,cost),'same_cost_own_state'))
for left,right in [(0,2),(0,5),(2,5)]:pairs.append(compare(('C_close_ROD_positive',left),('C_close_ROD_positive',right),'C_across_costs'))
pd.DataFrame(pairs).to_csv(card/'failure_path_divergence.csv',index=False)
held=[]
for (family,cost),book in books.items():
    path=book['authority']
    held.append({'family':family,'slippage_bps':cost,'mean_ending_SOXL_weight':float(path.ending_soxl_weight.mean()),
        'mean_ending_SOXX_weight':float(path.ending_soxx_weight.mean()),'mean_ending_cash_weight':float(path.ending_cash_weight.mean()),
        'mean_ending_nominal_beta':float((3*path.ending_soxl_weight+path.ending_soxx_weight).mean()),
        'risk_state_nonzero_days':int(path.risk_state.ne(0).sum()),'open_execute_days':int(path.execute_open_target.sum()),
        'recovery_action_days':int(path.recovery_action.sum()),'basis':'Recorded next-open endpoint weights, not intraday time-average exposure'})
pd.DataFrame(held).to_csv(card/'failure_held_exposure.csv',index=False)
metrics=objects['analysis/metrics'];scale=[]
for row in cash_rows:
    cost=row['slippage_bps'];lookup=metrics.loc[metrics.slippage_bps.eq(cost)].set_index('family')
    difference=100000*(float(lookup.loc[row['family'],'return_pct'])-float(lookup.loc['A_active_research_replay','return_pct']))/100
    scale.append({'family':row['family'],'slippage_bps':cost,'terminal_NAV_difference_vs_A_usd':difference,
        'sum_executed_lot_cash_pnl_usd':row['net_recorded_lot_cash_pnl_usd'],'not_an_additive_causal_decomposition':True})
ci=objects['analysis/uncertainty']
result={'status':'one_bounded_failure_attribution_completed','scope_plan_sha256':hashlib.sha256((card/'failure_attribution_plan.json').read_bytes()).hexdigest(),
    'current_recorder_id':summary['recorder_id'],'current_manifest_sha256':hashlib.sha256((card/'verification_manifest.json').read_bytes()).hexdigest(),
    'current_artifacts':hashes,'frozen_A_source':a_binding,'old_A_trace_reconstructed':False,
    'cashbook_summary':cash_rows,'cash_vs_terminal_scales':scale,'period_comparisons':len(period_table),
    'period_summary':period_summary.to_dict('records'),'state_comparisons':pairs,'held_exposure':held,
    'recorded_inputs_counts':{'known_source_days':int(objects['inputs/late_inputs'].rod_known.sum()),
        'recorded_positive_sign_days':int((objects['inputs/late_inputs'].rod_known & objects['inputs/late_inputs'].rod_positive).sum()),
        'signs_recomputed':False},'fixed_CI_positive_count':int(ci.lower_bound_positive.sum()),'fixed_CI_total':len(ci),
    'new_fit':0,'new_replay':0,'CI_resampling':0,'new_source_HTTP':0,'sealed_Recorder_or_NAS_modified':False,
    'supported':['Real integer lots occurred in everyB/Ccost arm; allwereclosed with exactshare/cashflowidentities.',
      'The fixedC version failsfullwealth/stability/allrequiredCI gates whilepassing sampledDD/CVaR; do notrescueby sign/window/cap changes.',
      'Direct executedlot cashchanges are only thousands ofdollars, while wholeaccount differences canbemuchlarger andpolicy-state paths diverge. Theseare distinct recordeddescriptives, not a uniquecontrollercausal decomposition.'],
    'limits':['Allhistoricalmarketdatareused; no independentforward or historicalPIT claim.',
      'Rejectioncounts use thefirst recordedfailedpredicate andthearms ownstates; theyare not independentcounterfactualopportunities.',
      'Across-cost differences contain originalStage1 fee/state sensitivity aswellas newlatefeedback; theyarenot simplefee subtraction orsoleROD effects.',
      'Astate comes from originalfrozen nativeauthority. No missing Atrace, rejectedopportunity return, or unexecuted strategywasreconstructed.']}
(card/'failure_attribution.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n')
{'cashbook':cash_rows,'cash_vs_terminal_scales':scale,'same_cost_2bp':[x for x in pairs if x['left_cost']==x['right_cost']==2],
 'C_across_costs':[x for x in pairs if x['role']=='C_across_costs'],'CI_passed':int(ci.lower_bound_positive.sum())}


{'cashbook': [{'family': 'B_close_permission',
   'slippage_bps': 0,
   'sessions': 944,
   'recorded_source_ready_days': 908,
   'recorded_permission_pass_days_after_arm_gate': 142,
   'late_trade_days': 142,
   'entry_orders': 189,
   'exit_orders': 189,
   'entry_notional_usd': 6623262.295184135,
   'exit_notional_usd': 6623386.964138031,
   'entry_fee_usd': 223.8875586692095,
   'exit_fee_usd': 360.31757150183796,
   'slippage_usd': 0.0,
   'gross_executed_lot_price_pnl_usd': 124.66895389556885,
   'net_recorded_lot_cash_pnl_usd': -459.5361762757602,
   'maximum_cash_identity_error_usd': 5.2665427574538626e-11,
   'maximum_inventory_identity_error_shares': 0.0,
   'all_lots_closed': True,
   'executed_roundtrip_shares_SOXL': 12931,
   'executed_roundtrip_shares_SOXX': 18299},
  {'family': 'B_close_permission',
   'slippage_bps': 2,
   'sessions': 944,
   'recorded_source_ready_days': 908,
   'recorded_permission_pass_days_after_arm_gate': 207,
   'late_trade_days': 207,
   'entry_o

# 尾盘ROD新增受限恢复许可：固定结果

原f696+b472晨间链保留；B/C共享新增尾盘许可，C仅多严格float64价格方向门。0拟合、6新原生账户；12旧native参考只被动增加估值点，不能称作12次新native验证。

|方案|滑点bp|净收益%|收盘DD%|全采样点DD%|
|---|---:|---:|---:|---:|
|A_active_research_replay|0|229.686|-29.487|-29.717|
|A_active_research_replay|2|280.650|-27.060|-27.266|
|A_active_research_replay|5|41.521|-30.517|-31.539|
|B_close_permission|0|267.165|-29.175|-29.411|
|B_close_permission|2|137.206|-27.666|-27.870|
|B_close_permission|5|177.288|-27.900|-28.103|
|SOXX|0|380.588|-40.863|-42.926|
|SOXX|2|380.568|-40.867|-42.929|
|SOXX|5|380.538|-40.872|-42.935|
|strong9097|0|441.681|-26.812|-26.812|
|strong9097|2|434.269|-26.881|-26.881|
|strong9097|5|423.730|-26.956|-26.956|
|C_close_ROD_positive|0|183.479|-29.386|-29.617|
|C_close_ROD_positive|2|138.856|-27.269|-27.474|
|C_close_ROD_positive|5|29.057|-27.447|-27.652|
|additional27baa|0|485.498|-27.171|-27.171|
|additional27baa|2|477.826|-27.223|-27.223|
|additional27baa|5|466.073|-27.317|-27.317|

决定：not_promoted_fixed_close_ROD_v1。失败或未获得的门：net_relative_wealth_all_costs, complete_annual_70pct, rolling126_70pct, paired_block_lower_positive, fresh_fullminute_risk_qualified, qualified_independent_forward。

全部年/季度（含partial）、126日滚动及12项置信下界均保留，不择窗；费用情景各自NAV/风险/费用状态独立。新增整数lot按15:59精确退出，不重置财富。来源为条件历史snapshot，假设延迟并非实盘首收或auction成交证明，重复历史不是前向独立证据。

本版各成本均发生实际增量lot，机会集非空；固定经济门失败，独立评估已确认关闭。本结论仅针对本固定版本，不改符号、窗口、上限、时钟或退出参数挽救。

Recorder：`f1f70f58b59946feabe131ff8b3d9645`。来源结论独立保存在source_preflight/summary.json；稳定超额目标未完成。

固定结果与区间差异：C在0/2/5bp的累计净收益为183.479%/138.856%/29.057%；2bp B为137.206%、A为280.650%、SOXX为380.568%。C只在2bp略胜B，不能通过全部成本对照。12项预定置信下界全部失败；30%采样回撤和CVaR门通过，但全窗财富、年度和rolling门失败，版本关闭。

完整2024/2025年度，C的2bp净收益分别−8.050%/18.324%，相对SOXX财富0.783028/0.823463。计入两个partial年度后，2bp C胜B为3/4、胜A及SOXX均0/4；全部16季度分别6/16、3/16、6/16。完整14季度及partial2季度单列，全部成本的180项配对完整保留于failure_period_comparison.csv。

一次固定归因并非机会空集：B在0/2/5bp实际尾盘交易日142/207/168，C为92/113/141。所有执行整数lot均精确平掉，累计股数恒等式误差0，现金恒等式最大误差约5.27e-11美元。2bp C已执行lot毛价差−158.71美元，entry+exit费用1939.72美元，现金净变动−2098.43美元；其对A终值差却为−141794.37美元。这是不同层面的已记录数值，不能把后者简单归于这笔费用或单一controller。

2bp C与B的risk_state/risk_clock在944日完全相同，但open执行决定18日不同，首次为2023-08-16。C与A的risk_state在241日不同，首日2024-02-21；clock不同274日，open执行不同58日。C三成本0对2、0对5、2对5之间，risk_state分别有170、304、194日不同。跨成本差异包括原Stage1自身费用/状态敏感性和新增尾盘反馈，不是简单费用相减。

2bp期末端点平均SOXL/SOXX/cash权重：C为12.388%/35.777%/51.835%，B为12.264%/35.646%/52.089%，A为15.082%/36.597%/48.321%。这是next-open端点持仓的描述，不是日内时间平均暴露或因果cash-drag分解。机会拒绝原因按原固定顺序记录第一个失败条件，且来自各自状态；详见failure_late_reasons.csv，不将其解释成共同反事实机会集。

一次交付冷读已由研究owner完成，见[readback_verification.json](readback_verification.json)：82个artifact、sign Dataset及6native/12passive角色核验。执行者没有重复NAS冷读或替自己签署晋升；固定负结果仍交独立结果评估。一次归因来源和公式见[failure_attribution.json](failure_attribution.json)。整个稳定超额Goal仍未达成。

独立评估：`ACCEPT_NEGATIVE_RESULT_CLOSE_FIXED_VERSION`。本固定版本正式关闭，不改符号、窗口、上限、退出或控制器挽救；持续Goal转入独立的辅助梯度投影训练设计，尚未达到稳定SOXX超额。见independent_evaluation.json。
